In [ ]:
%pip install -q langchain langchain-aws langgraph boto3 python-dotenv

# Week 6 Thursday -- Human-in-the-Loop & Guardrails

## Overview

Yesterday we built **multi-agent systems** where specialized agents collaborate on complex tasks. Today we shift focus to **safety and control**: how do we ensure those agents behave responsibly?

Two critical concerns emerge when agents act autonomously:

1. **Human-in-the-Loop (HITL)**: Some operations -- deleting data, sending emails, making purchases -- are too consequential to let an agent execute without supervision. We need the ability to pause, review, and approve before proceeding.

2. **Guardrails**: Even when an agent has full autonomy, we still want safety nets. Input validation catches harmful prompts before they reach the model. Output filtering catches PII or dangerous content before it reaches the user.

LangChain v1.0 gives us elegant tools for both: **lifecycle hooks** (`@before_model`, `@after_model`, `@before_tools`, `@after_tools`) for cross-cutting middleware, **interrupt/resume** for approval workflows, and **guardrail nodes** in StateGraph for structural safety.

## Learning Objectives

By the end of today, you will be able to:

- Explain the four lifecycle hook decorators and when to use each one
- Implement `@before_model` for input validation and PII redaction
- Implement `@after_model` for output filtering and stop-word detection
- Use `interrupt()` and `Command(resume=...)` for human approval workflows
- Design the approve / edit / reject pattern for dangerous operations
- Build guardrail nodes in a StateGraph for input/output content moderation
- Detect and redact PII using regex-based pattern matching
- Combine hooks and guardrails for comprehensive agent safety

## Today's Roadmap

| Stage | Topic | Demo | Readings |
|-------|-------|------|----------|
| 1 | Lifecycle Context Hooks | demo_02 | `@before_model`, `@after_model`, `@before_tools`, `@after_tools` |
| 2 | Human-in-the-Loop Patterns & Breakpoints | demo_01 | HITL patterns, interrupt/resume, breakpoints, approvals |
| 3 | Content Moderation & Guardrails | demo_03 | PII detection, safety rules, guardrail nodes in StateGraph |

---

# Stage 1: Lifecycle Context Hooks

Some concerns apply to **every** agent operation, regardless of what the agent does:

- **Logging** every model call for debugging
- **Validating** inputs before they reach the model
- **Filtering** outputs for content moderation
- **Authorizing** tool calls before execution

Rather than embedding these checks inside every agent, LangChain v1.0 provides **lifecycle hooks** -- middleware decorators that intercept execution at four key points in the agent loop.

## The Four Hooks

```
User Request
     │
     ▼
┌─────────────────┐
│  @before_model  │ ◄── Validate/preprocess before LLM
└────────┬────────┘
         │
         ▼
    [LLM Call]
         │
         ▼
┌─────────────────┐
│  @after_model   │ ◄── Process/filter after LLM
└────────┬────────┘
         │
    (If tool calls)
         │
         ▼
┌─────────────────┐
│  @before_tools  │ ◄── Authorize before tools
└────────┬────────┘
         │
         ▼
   [Tool Execution]
         │
         ▼
┌─────────────────┐
│  @after_tools   │ ◄── Handle errors after tools
└─────────────────┘
```

Each hook receives the current agent state and can either **modify** it (return a dict) or **pass through** (return `None`). Hooks are composable -- you can stack multiple `@before_model` hooks and they all run in sequence.

## `@before_model` — Input Preprocessing

The `@before_model` decorator runs **before every LLM call**. Common use cases:

- **Input validation**: reject messages that are too long, empty, or contain prohibited content
- **PII redaction**: strip sensitive data before it reaches the model
- **Message trimming**: manage context window by keeping only recent messages
- **Logging**: record every request for debugging

The hook signature is `def hook(state: AgentState, runtime: Runtime) -> dict | None`. Returning a dict updates the state; returning `None` means no changes.

## `@after_model` — Response Processing

The `@after_model` decorator runs **after every LLM response**. Common use cases:

- **Output filtering**: redact PII from the model's response
- **Stop-word detection**: block responses containing sensitive terms like passwords or API keys
- **Format validation**: ensure the response meets structural requirements
- **Metrics collection**: log response times and token usage

You can remove and replace messages using `RemoveMessage` -- this lets you swap out a problematic response with a sanitized version.

## `@before_tools` — Authorization

The `@before_tools` decorator runs **before tool execution**. It receives the list of pending tool calls and returns only the authorized ones:

- **Role-based access control**: restrict which tools each user role can invoke
- **Audit logging**: record every tool call for compliance
- **Input sanitization**: validate tool arguments before execution

## `@after_tools` — Error Handling

The `@after_tools` decorator runs **after tools return results**:

- **Error recovery**: distinguish retryable vs. fatal errors
- **Result validation**: truncate excessively long results
- **Audit completion**: log what each tool returned

Now let's see these hooks in action.

In [ ]:
import os
import re
from typing import Any
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model, after_model
from langchain.messages import RemoveMessage
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.runtime import Runtime
from langgraph.graph.message import REMOVE_ALL_MESSAGES

print("Stage 1 imports loaded.")

In [ ]:
# ---------------------------------------------------------------------------
# PII Detection Utilities
# These will be shared by our @before_model and @after_model hooks.
# ---------------------------------------------------------------------------

PII_PATTERNS = {
    "email": r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
    "phone": r'\b(?:\+?1[-.\s]?)?\(?[0-9]{3}\)?[-.\s]?[0-9]{3}[-.\s]?[0-9]{4}\b',
    "ssn": r'\b\d{3}[-.\s]?\d{2}[-.\s]?\d{4}\b',
    "credit_card": r'\b(?:\d{4}[-.\s]?){3}\d{4}\b',
}

HARMFUL_KEYWORDS = [
    "hack", "exploit", "bypass security", "steal password",
    "inject sql", "ddos", "malware", "ransomware",
]


def detect_pii(text: str) -> list[tuple[str, str]]:
    findings = []
    for pii_type, pattern in PII_PATTERNS.items():
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            findings.append((pii_type, match))
    return findings


def redact_pii(text: str) -> str:
    redacted = text
    for pii_type, pattern in PII_PATTERNS.items():
        redacted = re.sub(pattern, f"[REDACTED_{pii_type.upper()}]", redacted, flags=re.IGNORECASE)
    return redacted


def contains_harmful_content(text: str) -> list[str]:
    text_lower = text.lower()
    return [kw for kw in HARMFUL_KEYWORDS if kw in text_lower]


# Quick test
sample = "Contact me at john@example.com or 555-123-4567"
print("PII found:", detect_pii(sample))
print("Redacted :", redact_pii(sample))

### PART 1 — `@before_model` Hooks

We define two `@before_model` hooks:

1. **`validate_and_sanitize_input`** — checks for harmful content and redacts PII *before* the message reaches the LLM.
2. **`trim_conversation_history`** — keeps only the most recent messages to prevent context-window overflow in long conversations.

In [ ]:
@before_model
def validate_and_sanitize_input(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    messages = state.get("messages", [])
    if not messages:
        return None

    last_msg = messages[-1]
    content = last_msg.content if hasattr(last_msg, "content") else str(last_msg)

    harmful = contains_harmful_content(content)
    if harmful:
        print(f"[before_model] BLOCKED: {harmful}")
        return {
            "messages": [
                AIMessage(content=(
                    "I cannot process this request as it contains potentially harmful content. "
                    f"Detected keywords: {harmful}. Please rephrase your request."
                ))
            ]
        }

    pii_found = detect_pii(content)
    if pii_found:
        redacted_content = redact_pii(content)
        print(f"[before_model] PII REDACTED: {[p[0] for p in pii_found]}")
        return {"messages": [HumanMessage(content=redacted_content)]}

    print("[before_model] Input validated: OK")
    return None


@before_model
def trim_conversation_history(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    messages = state.get("messages", [])
    MAX_MESSAGES = 10

    if len(messages) <= MAX_MESSAGES:
        return None

    first_msg = messages[0]
    recent_messages = messages[-(MAX_MESSAGES - 1):]

    print(f"[before_model] Trimmed: {len(messages)} -> {len(recent_messages) + 1} messages")

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            first_msg,
            *recent_messages
        ]
    }

print("@before_model hooks defined.")

### PART 2 — `@after_model` Hooks

Two `@after_model` hooks:

1. **`validate_and_sanitize_output`** — scans every AI response for PII and redacts it before it reaches the user.
2. **`filter_stop_words`** — blocks responses containing sensitive terms like `password`, `secret_key`, or `api_key`.

In [ ]:
@after_model
def validate_and_sanitize_output(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    messages = state.get("messages", [])
    if not messages:
        return None

    last_msg = messages[-1]
    if not isinstance(last_msg, AIMessage):
        return None

    content = last_msg.content
    pii_found = detect_pii(content)
    if pii_found:
        redacted_content = redact_pii(content)
        print(f"[after_model] Output PII REDACTED: {[p[0] for p in pii_found]}")
        return {
            "messages": [
                RemoveMessage(id=last_msg.id),
                AIMessage(content=redacted_content)
            ]
        }

    print("[after_model] Output validated: OK")
    return None


@after_model
def filter_stop_words(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    STOP_WORDS = ["password", "secret_key", "api_key", "private_key"]

    messages = state.get("messages", [])
    if not messages:
        return None

    last_msg = messages[-1]
    if not isinstance(last_msg, AIMessage):
        return None

    content_lower = last_msg.content.lower()
    found_stop_words = [word for word in STOP_WORDS if word in content_lower]

    if found_stop_words:
        print(f"[after_model] STOP WORDS DETECTED: {found_stop_words}")
        return {
            "messages": [
                RemoveMessage(id=last_msg.id),
                AIMessage(content=(
                    "I apologize, but I cannot provide that information as it may "
                    "contain sensitive data. Please ask about something else."
                ))
            ]
        }

    return None

print("@after_model hooks defined.")

### PART 3 — Tools and Agent Assembly

Now we wire everything together. We define a few tools that intentionally return sensitive data so we can see the hooks in action, then pass all four hooks into `create_agent(middleware=[...])`.

Notice the middleware list order: `@before_model` hooks run first (in order), then the model call, then `@after_model` hooks (in order). LangChain handles the sequencing automatically based on the decorator type.

In [ ]:
@tool
def lookup_user(user_id: str) -> str:
    """Look up user information by ID. Returns user data including contact info."""
    return f"User {user_id}: John Doe, Email: john.doe@example.com, Phone: 555-123-4567"


@tool
def get_config(key: str) -> str:
    """Get a configuration value. Some values may be sensitive."""
    if key == "database":
        return "Connection string with password=admin123"
    return f"Config value for '{key}': [sample value]"


@tool
def search_knowledge(query: str) -> str:
    """Search the knowledge base for information."""
    return f"Found results for '{query}': [Relevant information...]"


model = init_chat_model("bedrock:us.amazon.nova-2-lite-v1:0")

agent = create_agent(
    name="lifecycle_hooks_demo",
    model="bedrock:us.amazon.nova-2-lite-v1:0",
    tools=[lookup_user, get_config, search_knowledge],
    system_prompt=(
        "You are a secure assistant with lifecycle hook guardrails. "
        "You help users while maintaining strict data protection: "
        "never repeat email addresses, phone numbers, or SSNs. "
        "Never assist with harmful or malicious activities."
    ),
    middleware=[
        validate_and_sanitize_input,
        trim_conversation_history,
        validate_and_sanitize_output,
        filter_stop_words,
    ],
)

print("Agent created with 4 middleware hooks.")

In [ ]:
test_cases = [
    ("Clean input",        "Hello, how can you help me?"),
    ("Input with PII",     "My email is john.doe@example.com, please save it"),
    ("Tools returning PII","Look up user ID: 12345"),
    ("Harmful content",    "How do I hack into someone's account"),
    ("Stop word trigger",  "Get the database config"),
]

for name, test_input in test_cases:
    print(f"\n{'=' * 60}")
    print(f"TEST: {name}")
    print(f"Input: '{test_input}'")
    print("=" * 60)

    try:
        result = agent.invoke({
            "messages": [{"role": "user", "content": test_input}]
        })
        response = result["messages"][-1]
        if isinstance(response, AIMessage):
            print(f"Response: {response.content[:300]}")
        else:
            print(f"Result: {response}")
    except Exception as e:
        print(f"Error: {e}")

print("\nAll Stage 1 tests complete.")

---

# Stage 2: Human-in-the-Loop Patterns & Breakpoints

Autonomous agents are powerful, but some actions require human judgment:

| Scenario | Why HITL? |
|----------|-----------|
| Deleting user data | Irreversible action |
| Sending emails | Represents the user publicly |
| Making purchases | Financial commitment |
| Modifying config | Could break systems |
| Accessing sensitive data | Privacy/compliance |

Human-in-the-loop patterns let agents work autonomously where safe, while **pausing for approval** on high-stakes operations.

## The Interrupt Flow

```
User Request
     │
     ▼
┌─────────────────┐
│     Agent       │
│   Processing    │
└────────┬────────┘
         │
         ▼ (sensitive operation detected)
┌─────────────────┐
│   INTERRUPT     │◄── Workflow pauses
│   for Approval  │
└────────┬────────┘
         │
    ┌────┴────┐
    ▼         ▼
┌───────┐ ┌───────┐ ┌───────┐
│Approve│ │ Edit  │ │Reject │
└───┬───┘ └───┬───┘ └───┬───┘
    │         │         │
    ▼         ▼         ▼
  Execute   Modified  Cancel
            Execute   Operation
```

## The Three Decision Types

1. **Approve**: Execute the operation as proposed
2. **Edit**: Modify parameters, then execute
3. **Reject**: Cancel the operation entirely

## Breakpoints & Conditional Interrupts

Not every invocation of the same tool needs approval. `HumanInTheLoopMiddleware` supports:

- **Per-tool rules**: `interrupt_on={"send_email": True, "search": False}`
- **Conditional logic**: a callable that inspects tool name *and* arguments — e.g., only interrupt purchases over $100
- **Streaming compatibility**: stream normally until an interrupt, then pause for a decision

A **checkpointer** is always required for HITL because state must persist between the interrupt and the resume.

In this demo we use LangGraph's `interrupt()` function and `Command(resume=...)` directly in a StateGraph to see the mechanics up close.

### Key APIs

- `interrupt(payload)` — pauses the graph and surfaces the payload to the caller
- `Command(resume={...})` — resumes execution, passing the human's decision back into the interrupted node
- `Command(goto="node_name", update={...})` — resume and route to a specific next node

The resume payload is returned by `interrupt()` inside the node function — so the node picks up exactly where it left off with the human's answer.

In [ ]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import interrupt, Command
from langchain_core.messages import BaseMessage


class HITLState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    action: str
    details: dict
    status: Literal["pending", "approved", "rejected", "completed"]


def receive_request(state: HITLState) -> dict:
    messages = state.get("messages", [])
    if not messages:
        return {"status": "pending"}

    last_msg = messages[-1]
    content = last_msg.content if hasattr(last_msg, "content") else str(last_msg)
    content_lower = content.lower()

    if "delete" in content_lower:
        action, details = "delete", {"target": content, "risk": "HIGH"}
    elif "write" in content_lower or "create" in content_lower:
        action, details = "write", {"target": content, "risk": "MEDIUM"}
    elif "execute" in content_lower or "run" in content_lower:
        action, details = "execute", {"command": content, "risk": "HIGH"}
    else:
        action, details = "safe", {"query": content, "risk": "LOW"}

    print(f"[receive_request] action={action}, risk={details.get('risk')}")
    return {
        "action": action,
        "details": details,
        "status": "pending",
        "messages": [AIMessage(content=f"Request received: {action} operation")]
    }


def check_if_dangerous(state: HITLState) -> Literal["request_approval", "execute_safe"]:
    action = state.get("action", "safe")
    route = "request_approval" if action in ["delete", "write", "execute"] else "execute_safe"
    print(f"[check_if_dangerous] routing to: {route}")
    return route

print("State and routing nodes defined.")

In [ ]:
def request_approval(state: HITLState) -> Command[Literal["execute_approved", "handle_rejection"]]:
    action = state.get("action", "unknown")
    details = state.get("details", {})

    # execution PAUSES here -- payload surfaces in result["__interrupt__"]
    response = interrupt({
        "message": f"APPROVAL REQUIRED: {action.upper()} operation",
        "action": action,
        "details": details,
        "instructions": {
            "approve": '{"action": "approve"}',
            "approve_with_edit": '{"action": "approve", "details": {"target": "new_value"}}',
            "reject": '{"action": "reject"}'
        }
    })

    # after resume -- response contains the human's decision
    if response.get("action") == "approve":
        edited_details = response.get("details", details)
        print(f"[request_approval] APPROVED with details: {edited_details}")
        return Command(
            goto="execute_approved",
            update={
                "details": edited_details,
                "status": "approved",
                "messages": [AIMessage(content="Operation approved by user")]
            }
        )
    else:
        print("[request_approval] REJECTED")
        return Command(
            goto="handle_rejection",
            update={
                "status": "rejected",
                "messages": [AIMessage(content="Operation rejected by user")]
            }
        )


def execute_approved(state: HITLState) -> dict:
    action = state.get("action", "unknown")
    details = state.get("details", {})
    result = f"Successfully executed {action}: {details}"
    print(f"[execute_approved] {result}")
    return {"status": "completed", "messages": [AIMessage(content=result)]}


def execute_safe(state: HITLState) -> dict:
    query = state.get("details", {}).get("query", "unknown request")
    result = f"Safe operation completed: {query}"
    print(f"[execute_safe] {result}")
    return {"status": "completed", "messages": [AIMessage(content=result)]}


def handle_rejection(state: HITLState) -> dict:
    action = state.get("action", "unknown")
    result = f"Operation '{action}' was cancelled. No changes made."
    print(f"[handle_rejection] {result}")
    return {"status": "rejected", "messages": [AIMessage(content=result)]}

print("Interrupt + execution nodes defined.")

In [ ]:
workflow = StateGraph(HITLState)

workflow.add_node("receive_request", receive_request)
workflow.add_node("request_approval", request_approval)
workflow.add_node("execute_approved", execute_approved)
workflow.add_node("execute_safe", execute_safe)
workflow.add_node("handle_rejection", handle_rejection)

workflow.add_edge(START, "receive_request")
workflow.add_conditional_edges(
    "receive_request",
    check_if_dangerous,
    {"request_approval": "request_approval", "execute_safe": "execute_safe"}
)
workflow.add_edge("execute_approved", END)
workflow.add_edge("execute_safe", END)
workflow.add_edge("handle_rejection", END)

from langgraph.checkpoint.memory import InMemorySaver

hitl_graph = workflow.compile(checkpointer=InMemorySaver())

print("HITL graph compiled.")
print()
print("TEST 1 — Safe operation (no interrupt)")
print("=" * 50)

config1 = {"configurable": {"thread_id": "test-safe"}}
result = hitl_graph.invoke({
    "messages": [HumanMessage(content="What is the weather today?")],
    "action": "",
    "details": {},
    "status": "pending"
}, config=config1)

print(f"Status  : {result['status']}")
print(f"Response: {result['messages'][-1].content}")

In [ ]:
print("TEST 2 — Dangerous operation (interrupt + approve)")
print("=" * 50)

config2 = {"configurable": {"thread_id": "test-dangerous"}}
result = hitl_graph.invoke({
    "messages": [HumanMessage(content="Delete all temporary files")],
    "action": "",
    "details": {},
    "status": "pending"
}, config=config2)

# The graph pauses at interrupt() -- check for __interrupt__ in state
state = hitl_graph.get_state(config2)
print(f"Interrupted: {bool(state.tasks)}")

if state.tasks:
    interrupts = [t.interrupts for t in state.tasks if t.interrupts]
    if interrupts:
        print(f"Interrupt payload: {interrupts[0][0].value}")

    # Resume with approval
    print("\nResuming with approval...")
    final = hitl_graph.invoke(
        Command(resume={"action": "approve"}),
        config=config2
    )
    print(f"Status  : {final['status']}")
    print(f"Response: {final['messages'][-1].content}")

print()
print("TEST 3 — Dangerous operation (interrupt + reject)")
print("=" * 50)

config3 = {"configurable": {"thread_id": "test-reject"}}
result = hitl_graph.invoke({
    "messages": [HumanMessage(content="Execute rm -rf /tmp/*")],
    "action": "",
    "details": {},
    "status": "pending"
}, config=config3)

state3 = hitl_graph.get_state(config3)
if state3.tasks:
    print("Resuming with rejection...")
    final3 = hitl_graph.invoke(
        Command(resume={"action": "reject"}),
        config=config3
    )
    print(f"Status  : {final3['status']}")
    print(f"Response: {final3['messages'][-1].content}")

---

# Stage 3: Content Moderation & Guardrails

Production agents must protect against:

- **Harmful content** generation
- **PII leakage** in responses
- **Policy violations** specific to your domain
- **Prompt injection** attacks

In Stage 1 we used `@before_model` / `@after_model` hooks — those are great when you use `create_agent`. But what if you're building a custom `StateGraph`? You can achieve the same protection by adding **guardrail nodes** directly into the graph.

## The Guardrail Layer

```
User Input
     │
     ▼
┌──────────────────┐
│ Input Guardrail  │◄── Block harmful inputs / redact PII
└────────┬─────────┘
         │ (not blocked)
         ▼
    [Agent Node]
         │
    ┌────┴────┐
    ▼ tools?  ▼ no
  [Tools]  [Output Guardrail] ◄── Redact PII from response
    │           │
    ▼           ▼
  [Agent]      END
```

The input guardrail can **short-circuit** the entire graph by routing directly to END when it detects harmful content. The output guardrail runs after the agent's final response, redacting any PII that slipped through.

This approach gives you **explicit control** over the flow — you can see the guardrail nodes in the graph visualization and reason about them just like any other node.

### Custom Domain Rules

You can create domain-specific guardrails for any industry:

- **Healthcare**: block requests to diagnose or prescribe
- **Finance**: flag transactions above a threshold
- **Legal**: prevent unauthorized disclosure of privileged information

The pattern is always the same: a guardrail node checks the content against your rules and either passes through, modifies the message, or blocks entirely.

In [ ]:
from typing import Annotated, Literal, TypedDict as TD
from langchain_core.messages import AnyMessage
from langgraph.prebuilt import ToolNode


class GuardrailState(TD):
    messages: Annotated[list[AnyMessage], add_messages]
    pii_detected: bool
    pii_redacted: bool
    blocked: bool
    block_reason: str


@tool
def lookup_user_gr(user_id: str) -> str:
    """Look up user information by ID."""
    return f"User {user_id}: John Doe, Email: john.doe@example.com, Phone: 555-123-4567"


@tool
def search_knowledge_gr(query: str) -> str:
    """Search the knowledge base."""
    return f"Found results for '{query}': [Relevant information...]"


@tool
def process_request_gr(request: str) -> str:
    """Process a general request."""
    return f"Request processed: {request}"


TOOLS_GR = [lookup_user_gr, search_knowledge_gr, process_request_gr]

gr_model = init_chat_model("bedrock:us.amazon.nova-2-lite-v1:0").bind_tools(TOOLS_GR)

print("Guardrail state, tools, and model ready.")

In [ ]:
def input_guardrail(state: GuardrailState) -> dict:
    messages = state.get("messages", [])
    if not messages:
        return {}

    last_msg = messages[-1]
    content = last_msg.content if hasattr(last_msg, "content") else str(last_msg)

    harmful = contains_harmful_content(content)
    if harmful:
        print(f"[input_guardrail] BLOCKED: {harmful}")
        return {
            "blocked": True,
            "block_reason": f"Harmful content detected: {harmful}",
            "messages": [AIMessage(content=(
                "I cannot process this request as it contains potentially harmful content. "
                f"Detected keywords: {harmful}. Please rephrase your request."
            ))]
        }

    pii_found = detect_pii(content)
    if pii_found:
        redacted_content = redact_pii(content)
        print(f"[input_guardrail] PII redacted: {[p[0] for p in pii_found]}")
        return {
            "pii_detected": True,
            "pii_redacted": True,
            "messages": [HumanMessage(content=redacted_content)]
        }

    print("[input_guardrail] Clean input, passing through.")
    return {"blocked": False, "pii_detected": False}


def should_continue_after_input(state: GuardrailState) -> Literal["agent", "end"]:
    if state.get("blocked", False):
        return "end"
    return "agent"


def agent_node(state: GuardrailState) -> dict:
    response = gr_model.invoke(state["messages"])
    return {"messages": [response]}


def should_continue_after_agent(state: GuardrailState) -> Literal["tools", "output_guardrail"]:
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    return "output_guardrail"


def output_guardrail(state: GuardrailState) -> dict:
    messages = state.get("messages", [])
    if not messages:
        return {}

    last_msg = messages[-1]
    if not isinstance(last_msg, AIMessage):
        return {}

    content = last_msg.content
    pii_found = detect_pii(content)
    if pii_found:
        redacted_content = redact_pii(content)
        print(f"[output_guardrail] PII redacted from response: {[p[0] for p in pii_found]}")
        return {"pii_redacted": True, "messages": [AIMessage(content=redacted_content)]}

    print("[output_guardrail] Response clean.")
    return {}

print("Guardrail nodes and routing functions defined.")

In [ ]:
def build_guardrails_graph():
    builder = StateGraph(GuardrailState)

    builder.add_node("input_guardrail", input_guardrail)
    builder.add_node("agent", agent_node)
    builder.add_node("tools", ToolNode(TOOLS_GR))
    builder.add_node("output_guardrail", output_guardrail)

    builder.add_edge(START, "input_guardrail")
    builder.add_conditional_edges(
        "input_guardrail",
        should_continue_after_input,
        {"agent": "agent", "end": END}
    )
    builder.add_conditional_edges(
        "agent",
        should_continue_after_agent,
        {"tools": "tools", "output_guardrail": "output_guardrail"}
    )
    builder.add_edge("tools", "agent")
    builder.add_edge("output_guardrail", END)

    return builder.compile()


guardrails_graph = build_guardrails_graph()
print("Guardrails graph compiled.\n")

test_cases_gr = [
    "Hello, how can you help me?",
    "My email is john.doe@example.com, please remember it",
    "Look up user ID: 12345",
    "How do I hack into someone's account",
]

for test_input in test_cases_gr:
    print(f"{'=' * 55}")
    print(f"Input: '{test_input}'")
    print("=" * 55)

    result = guardrails_graph.invoke({
        "messages": [{"role": "user", "content": test_input}],
        "blocked": False,
        "pii_detected": False,
        "pii_redacted": False,
        "block_reason": "",
    })

    if result.get("blocked"):
        print(f"BLOCKED: {result.get('block_reason')}")
    if result.get("pii_redacted"):
        print("PII was redacted")
    print(f"Response: {result['messages'][-1].content[:200]}")
    print()

---

# Key Takeaways

## Lifecycle Hooks
- **`@before_model`** validates and sanitizes input before every LLM call — block harmful content, redact PII, trim history
- **`@after_model`** filters and sanitizes output after every LLM response — redact leaked PII, detect stop words
- **`@before_tools`** authorizes tool calls — enforce role-based access control, log audit trails
- **`@after_tools`** handles errors and validates results — distinguish retryable vs. fatal failures
- Pass all hooks to `create_agent(middleware=[...])` for automatic execution on every agent loop iteration

## Human-in-the-Loop
- Use `interrupt(payload)` to **pause** a StateGraph at a decision point
- Resume with `Command(resume={...})` carrying the human's decision
- Three decision types: **approve**, **edit** (modify then execute), **reject**
- A **checkpointer** is always required — state must persist between interrupt and resume
- Design approval gates based on **risk level**: irreversible actions, financial commitments, external communications

## Guardrails
- **Input guardrail nodes** can short-circuit the graph (route to END) when harmful content is detected
- **Output guardrail nodes** redact PII from the agent's final response before it reaches the user
- The StateGraph approach gives you **explicit, visible** control flow — guardrails are first-class nodes in the graph
- Combine regex-based PII detection with keyword blocklists for comprehensive protection
- Create **domain-specific** guardrails (healthcare, finance, legal) using the same node pattern

## Hooks vs. Guardrail Nodes

| Approach | Best For | Mechanism |
|----------|----------|-----------|
| `@before_model` / `@after_model` | Cross-cutting concerns with `create_agent` | Middleware decorators |
| Guardrail Nodes in StateGraph | Custom graphs needing explicit flow control | Graph nodes with conditional edges |
| `HumanInTheLoopMiddleware` | Tool-level approval with `create_agent` | Middleware with interrupt |
| `interrupt()` / `Command(resume=...)` | Custom approval flows in StateGraph | Graph-level pause/resume |

---

## Up Next: Friday — Advanced Middleware & Long-Term Memory

Tomorrow we'll explore:
- **Advanced middleware patterns** — chaining multiple middleware layers, custom middleware classes
- **Long-term memory** — persisting agent knowledge across sessions using memory stores
- **Combining memory with guardrails** — ensuring remembered context is also safe and compliant